# P2 · FAQ y repaso

**Modelado documental vs analítico (JSON ↔ Parquet + DuckDB)**

Este cuaderno no sustituye al enunciado de la tarea. Reúne las dudas que más
aparecen en el laboratorio, las comprobaciones que conviene saber explicar y un
repaso corto antes de entregar.

**Entorno:** `sbd-lab`. Ejecuta las celdas en orden.


## Cómo usar este cuaderno

- Ejecuta las celdas de arriba abajo.
- Las celdas leen `data/orders_flat.csv`; no modifican los datos.
- Si ejecutas `p2_duckdb.sql` desde una celda, hazlo sobre una copia: crea una
  tabla y escribe `data/orders_flat.parquet`.
- Las preguntas del final son de repaso; no son una entrega nueva.


## Mapa del laboratorio

| Archivo | Qué representa | Granularidad |
|---|---|---|
| `data/orders.ndjson` | Un documento JSON por pedido, con `items` anidados | 1 registro = 1 pedido |
| `data/orders_flat.csv` | Vista plana para consultas | 1 fila = 1 línea de pedido |
| `p2_duckdb.sql` | Crea la tabla, exporta Parquet y lanza consultas | — |

La pregunta clave: **¿una fila es un pedido o una línea?** En el CSV es una
línea; un pedido puede ocupar varias filas.


## Materiales del laboratorio

Todo lo necesario está en la carpeta del laboratorio. Si te falta algo, usa el
ZIP publicado en Classroom y descomprímelo en una carpeta de trabajo.

| Archivo | Para qué sirve |
|---|---|
| `README.md` | Arranque y comandos básicos. |
| `genera_datos_p2.py` | Genera `orders.ndjson` y `orders_flat.csv`. |
| `p2_duckdb.sql` | Crea la tabla, exporta Parquet y lanza consultas. |
| `data/orders_flat.csv` | Vista plana: una fila por línea. |
| `data/orders.ndjson` | Documentos: un pedido por línea. |
| `data/orders_flat.parquet` | Salida columnar generada por el SQL. |
| `requirements.txt` | Dependencias con pip. |
| `environment.yml` y `README_entorno_conda.md` | Entorno conda `sbd-lab`. |
| `README_practica_modelado_completa.md` | Guía ampliada del laboratorio. |

MongoDB es una ampliación opcional: no es necesaria para entregar.


In [1]:
from pathlib import Path

requeridos = [
    "README.md",
    "genera_datos_p2.py",
    "p2_duckdb.sql",
    "data/orders_flat.csv",
    "data/orders.ndjson",
]
for r in requeridos:
    print("OK   " if Path(r).exists() else "FALTA", r)


OK    README.md
OK    genera_datos_p2.py
OK    p2_duckdb.sql
OK    data/orders_flat.csv
OK    data/orders.ndjson


In [2]:
import pandas as pd

df = pd.read_csv("data/orders_flat.csv")
print("Filas:", len(df))
print("Pedidos distintos:", df["order_id"].nunique())
print("Columnas:", df.columns.tolist())


Filas: 22579
Pedidos distintos: 9057
Columnas: ['order_id', 'customer_id', 'fecha', 'canal', 'sku', 'qty', 'price', 'cost']


## FAQ 1 · ¿Por qué hay más filas que pedidos?

Porque el CSV está **aplanado**: cada fila es una línea de pedido. `order_id` se
repite para relacionar las líneas del mismo pedido. No es un duplicado.

`COUNT(*)` cuenta líneas. Para contar pedidos: `COUNT(DISTINCT order_id)`.


In [3]:
df[df["order_id"] == 1]


,order_id,customer_id,fecha,canal,sku,qty,price,cost
0,1,2807,2024-06-01,web,SKU0083,1,20.48,11.66
1,1,2807,2024-06-01,web,SKU0018,4,7.03,5.35


En la copia actual, `order_id = 1` tiene dos líneas. Es **un pedido**, no dos.
Si el ejercicio pide «cuenta los pedidos», no uses `COUNT(*)` sobre el CSV.


## FAQ 2 · ¿Dónde está `importe`?

No está en el CSV original. Se calcula:

- `importe = qty * price`
- `margen_u = price - cost`
- `margen_total = (price - cost) * qty`

`p2_duckdb.sql` crea esas columnas al construir `orders_flat`.


In [4]:
df.assign(
    importe=df["qty"] * df["price"],
    margen_u=df["price"] - df["cost"],
).head()


,order_id,customer_id,fecha,canal,sku,qty,price,cost,importe,margen_u
0,1,2807,2024-06-01,web,SKU0083,1,20.48,11.66,20.48,8.82
1,1,2807,2024-06-01,web,SKU0018,4,7.03,5.35,28.12,1.68
2,2,768,2024-06-01,app,SKU0009,3,21.67,14.29,65.01,7.38
3,2,768,2024-06-01,app,SKU0027,1,12.20,8.83,12.20,3.37
4,3,1768,2024-06-01,tienda,SKU0087,3,6.49,4.43,19.47,2.06


## FAQ 3 · ¿Cómo compruebo el importe de 26,44 €?

En el ejemplo de papel:

- línea 1: `2 × 5,93 = 11,86 €`
- línea 2: `2 × 7,29 = 14,58 €`
- total: `11,86 + 14,58 = 26,44 €`

No compares `float` con `==`: usa una tolerancia.


In [5]:
import math

total = 2 * 5.93 + 2 * 7.29
print("Total:", total)
print("¿Coincide con 26,44?", math.isclose(total, 26.44, abs_tol=0.005))


Total: 26.439999999999998
¿Coincide con 26,44? True


Ese ejemplo no tiene por qué aparecer en el CSV actual: el generador puede
haberse ejecutado de nuevo. Sirve para comprobar la **lógica de cálculo**, no
para buscar una fila concreta.


## FAQ 4 · ¿Cómo ejecuto las consultas?

Desde la carpeta del laboratorio:

```bash
duckdb -c ".read p2_duckdb.sql"
```

`.read` es un comando de la CLI de DuckDB, no de Python. Si trabajas en el
cuaderno, puedes consultar el CSV directamente:


In [6]:
import duckdb

consulta_ventas = """
SELECT canal,
       strftime(fecha, '%Y-%m') AS ym,
       ROUND(SUM(qty * price), 2) AS venta
FROM read_csv_auto('data/orders_flat.csv')
GROUP BY 1, 2
ORDER BY 2, 3 DESC
"""

duckdb.sql(consulta_ventas).df().head(10)


,canal,ym,venta
0,app,2024-06,97240.95
1,web,2024-06,96931.89
2,tienda,2024-06,95250.52
3,web,2024-07,95297.95
4,app,2024-07,92071.47
5,tienda,2024-07,91172.08


In [7]:
consulta_margen = """
SELECT sku,
       ROUND(SUM((price - cost) * qty), 2) AS margen_total
FROM read_csv_auto('data/orders_flat.csv')
GROUP BY 1
ORDER BY margen_total DESC
LIMIT 5
"""

duckdb.sql(consulta_margen).df()


,sku,margen_total
0,SKU0067,2478.17
1,SKU0051,2376.53
2,SKU0069,2305.91
3,SKU0078,2303.89
4,SKU0084,2299.98


## FAQ 5 · ¿Qué hace exactamente `p2_duckdb.sql`?

1. Lee `data/orders_flat.csv`.
2. Crea la tabla `orders_flat` con `importe` y `margen_u`.
3. Exporta la tabla a `data/orders_flat.parquet`.
4. Lanza tres consultas: ventas por canal/mes, top 5 de margen y clientes con
   más de 5 pedidos.

Si lo ejecutas desde el cuaderno, hazlo en una copia: el `COPY` sobrescribe el
Parquet.


In [8]:
from pathlib import Path

sql = Path("p2_duckdb.sql").read_text(encoding="utf-8")
print(sql[:400])
# No ejecutes el contenido aquí sin una copia: escribe en data/.
# con.execute(sql)


INSTALL parquet; LOAD parquet;
CREATE OR REPLACE TABLE orders_flat AS
SELECT *, qty*price AS importe, price - cost AS margen_u
FROM read_csv('data/orders_flat.csv', AUTO_DETECT=TRUE);
COPY (SELECT * FROM orders_flat) TO 'data/orders_flat.parquet' (FORMAT 'parquet');
SELECT canal, strftime(fecha, '%Y-%m') AS ym, ROUND(SUM(importe),2) AS venta
FROM orders_flat GROUP BY 1,2 ORDER BY 2,3 DESC;
SELECT 


## FAQ 6 · Errores frecuentes

| Síntoma | Causa probable | Qué hacer |
|---|---|---|
| `IO Error: No files found` | Estás fuera de la carpeta del laboratorio | `cd` a `lab_p2_modelado_documental_analitico/` |
| `duckdb: command not found` | CLI no instalada | Usa `import duckdb` desde Python o el entorno `sbd-lab` |
| `importe` no existe en el CSV | Es una columna calculada | Créala con `qty * price` |
| `price = 26.44` no devuelve nada | Comparación exacta de `float` | Usa `abs(price - 26.44) < 0.005` |
| El Parquet cambia o se sobrescribe | `COPY` del script | Ejecuta sobre una copia |


## FAQ 7 · ¿JSON o CSV?

- **NDJSON/documental:** un documento por pedido, con `items` anidados. Bueno
  para recuperar el pedido completo.
- **CSV/Parquet/analítico:** una fila por línea. Bueno para agregar ventas y
  márgenes.

No es «uno mejor que otro»: depende de cómo lees. En la tarea debes justificarlo.
MongoDB es opcional; no es requisito para entregar.


## Limitaciones que debes poder explicar

- El CSV no tiene `line_number`: no hay una clave de línea perfecta.
- No hay catálogo de clientes ni de productos.
- Los datos son sintéticos.
- `ts` no lleva zona horaria.
- Una prueba con estos datos no demuestra rendimiento ni escalabilidad.


## Mini-repaso

Responde sin mirar las celdas anteriores:

1. ¿Cuántos pedidos hay en el CSV? ¿Qué consulta usarías?
2. ¿Por qué `order_id` se repite?
3. ¿Qué diferencia hay entre `importe` y `margen_total`?
4. ¿Por qué no se debe comparar un `float` con `==`?
5. ¿Qué gana Parquet frente a CSV y qué no demuestra la prueba?


In [9]:
# Escribe aquí tus respuestas o tus comprobaciones.


## Checklist antes de entregar

- [ ] El ZIP contiene `respuestas.md`, `p2_duckdb.sql` y `README.md`.
- [ ] El `README.md` indica los dos nombres y los comandos.
- [ ] El ejemplo JSON se entiende sin abrir otros archivos.
- [ ] Las consultas se ejecutan sin errores sobre una copia.
- [ ] La consulta adicional está justificada.
- [ ] Se explican las limitaciones y no se prometen tiempos.


## Rúbrica (/10)

| Criterio | Puntos |
|---|---:|
| Estructura, tipologías y campos del documento JSON | 2 |
| Uso y límites del modelo documental | 2 |
| Consultas DuckDB correctas, legibles y reproducibles | 3 |
| Comparación JSON/documental y Parquet/analítico | 2 |
| README y entrega ordenados | 1 |
